# Design and Empirical Study of a Large Language Model-Based Multi-Agent Investment System for Chinese Public REITs

**Authors:** Zheng Li
**Published:** 2026-01-22
**Arxiv:** [https://arxiv.org/abs/2602.00082](https://arxiv.org/abs/2602.00082)

**Abstract:**
This study addresses the low-volatility Chinese Public Real Estate Investment Trusts (REITs) market, proposing a large language model (LLM)-driven trading framework based on multi-agent collaboration. The system constructs four types of analytical agents—announcement, event, price momentum, and market—each conducting analysis from different dimensions; then the prediction agent integrates these multi-source signals to output directional probability distributions across multiple time horizons, then the decision agent generates discrete position adjustment signals based on the prediction results and risk control constraints, thereby forming a closed loop of analysis-prediction-decision-execution. This study further compares two prediction model pathways: for the prediction agent, directly calling the general-purpose large model DeepSeek-R1 versus using a specialized small model Qwen3-8B fine-tuned via supervised fine-tuning and reinforcement learning alignment. In the backtest from October 2024 to October 2025, both agent-based strategies significantly outperformed the buy-and-hold benchmark in terms of cumulative return, Sharpe ratio, and maximum drawdown. The results indicate that the multi-agent framework can effectively enhance the risk-adjusted return of REITs trading, and the fine-tuned small model performs close to or even better than the general-purpose large model in some scenarios.

In [ ]:
!pip install yfinance pandas numpy matplotlib scipy

## Phase 1 — Trading Context & Objectives

In [ ]:
# Configuration

# Define the universe of tickers
UNIVERSE = ['AAPL', 'MSFT']

# Define parameters
START_DATE = '2020-01-01'
END_DATE = '2023-01-01'

# Hypothesis
# The hypothesis is that a multi-agent system leveraging large language models can generate superior risk-adjusted returns in the REITs market by integrating various analytical signals.

## Phase 2 — Data Download & Feature Computation

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np

# Download data
data = yf.download(UNIVERSE, start=START_DATE, end=END_DATE)

# Compute features
data['Returns'] = data['Adj Close'].pct_change()
data['Volatility'] = data['Returns'].rolling(window=21).std()
data['Momentum'] = data['Returns'].rolling(window=126).mean()

# Cross-sectional normalization
data['Normalized_Momentum'] = data['Momentum'].rank(axis=1, pct=True)

## Phase 3 — Signal Generation & Portfolio Construction

In [ ]:
# Signal generation
data['Signal'] = np.where(data['Normalized_Momentum'] > 0.5, 1, -1)

# Position sizing
data['Position'] = data['Signal'].shift(1)

# Portfolio construction
data['Portfolio_Return'] = data['Returns'] * data['Position']

## Phase 4 — Vectorized Backtest

In [ ]:
# Calculate cumulative returns
data['Cumulative_Return'] = (1 + data['Portfolio_Return']).cumprod()

# Plot equity curve
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
plt.plot(data['Cumulative_Return'], label='Strategy')
plt.plot((1 + data['Returns'].mean()).cumprod(), label='Buy and Hold')
plt.legend()
plt.show()

## Phase 5 — Performance Metrics

In [ ]:
from scipy.stats import norm

# Calculate performance metrics
annual_return = data['Portfolio_Return'].mean() * 252
annual_volatility = data['Portfolio_Return'].std() * np.sqrt(252)
sharpe_ratio = annual_return / annual_volatility
sortino_ratio = annual_return / data['Portfolio_Return'][data['Portfolio_Return'] < 0].std() * np.sqrt(252)
max_drawdown = (data['Cumulative_Return'].cummax() - data['Cumulative_Return']).max()
calmar_ratio = annual_return / max_drawdown

print(f'Annual Return: {annual_return:.2%}')
print(f'Annual Volatility: {annual_volatility:.2%}')
print(f'Sharpe Ratio: {sharpe_ratio:.2f}')
print(f'Sortino Ratio: {sortino_ratio:.2f}')
print(f'Max Drawdown: {max_drawdown:.2%}')
print(f'Calmar Ratio: {calmar_ratio:.2f}')

## Phase 6 — Monitoring Stub

In [ ]:
# Function to print daily P&L and current positions
def monitor_positions(data):
    latest_date = data.index[-1]
    latest_positions = data.loc[latest_date, 'Position']
    latest_pnl = data.loc[latest_date, 'Portfolio_Return']
    
    print(f'Date: {latest_date}')
    print(f'Daily P&L: {latest_pnl:.2%}')
    print('Current Positions:')
    print(latest_positions)}

# Example usage
monitor_positions(data)